# Lab 01 - Local Evaluation (starter)

Fill in the `TODO` blocks. The complete version is in `lab01_local_evaluation_solution.ipynb` -
use it only after trying, or to catch up if you fall behind.

> **Judge model:** local agent evaluators in `azure-ai-evaluation 1.18.3` send the legacy `max_tokens`
> parameter, so they must use the `gpt-4.1-mini` class deployment configured in
> `AZURE_OPENAI_EVALUATION_COMPATIBLE_DEPLOYMENT_NAME`.


## Step 0 - Configuration (ready to run)

In [ ]:
import sys, json, warnings
from pprint import pprint

sys.path.append("assets")          # friend.py / response_length_score.py live here
warnings.filterwarnings("ignore")

from lab_utils import load_settings

settings = load_settings(verbose=True)

In [ ]:
from azure.ai.evaluation import AzureOpenAIModelConfiguration

model_config = AzureOpenAIModelConfiguration(
    azure_endpoint=settings["azure_openai_endpoint"],
    azure_deployment=settings["azure_evaluation_compatible_deployment_name"],
    api_version=settings["openai_api_version"],
)

credential = settings["credential"]
model_config["azure_deployment"]

## Step 1 - Intent Resolution on plain strings (~8 min)

1. Import and instantiate `IntentResolutionEvaluator` with `model_config` and `credential`.
2. Score one *good* pair and one *bad* pair and compare `intent_resolution`.

In [ ]:
from azure.ai.evaluation import IntentResolutionEvaluator

# TODO 1.1 - create the evaluator
intent_resolution_evaluator = ...

# TODO 1.2 - a response that fully resolves the intent
good = intent_resolution_evaluator(
    query="What are the opening hours of the Eiffel Tower?",
    response=...,
)
pprint(good)

# TODO 1.3 - a response that does NOT resolve the intent
bad = intent_resolution_evaluator(
    query="What are the opening hours of the Eiffel Tower?",
    response=...,
)
pprint(bad)

## Step 2 - Evaluate a conversation loaded from disk (~10 min)

`assets/sample_synthetic_conversations.jsonl` holds 90 records with `messages` and `tools`.
Split the conversation at the **last user message**: everything up to it is the `query`,
everything after it is the `response`.

In [ ]:
def load_conversations(filename):
    with open(filename, "r", encoding="utf-8") as file:
        conversations = [json.loads(line) for line in file if line.strip()]
    print(f"Loaded {len(conversations)} conversations from {filename}.")
    return conversations


conversations = load_conversations("assets/sample_synthetic_conversations.jsonl")
conversation = conversations[10]
messages = conversation["messages"]

# TODO 2.1 - index of the LAST message whose role is "user"
last_user_index = ...

# TODO 2.2 - build query / response / tool_definitions
query = ...
response = ...
tool_definitions = ...

# TODO 2.3 - evaluate
pprint(intent_resolution_evaluator(query=query, response=response, tool_definitions=tool_definitions))

## Step 3 - Tool Call Accuracy and Task Adherence (~10 min)

Add a second tool call for a location the user never mentioned and watch the passing rate drop.

In [ ]:
from azure.ai.evaluation import ToolCallAccuracyEvaluator, TaskAdherenceEvaluator

weather_tool = {
    "id": "fetch_weather",
    "name": "fetch_weather",
    "description": "Fetches the weather information for the specified location.",
    "parameters": {
        "type": "object",
        "properties": {"location": {"type": "string", "description": "The location to fetch weather for."}},
    },
}

single_call = {
    "type": "tool_call",
    "tool_call_id": "call_1",
    "name": "fetch_weather",
    "arguments": {"location": "Seattle"},
}

# TODO 3.1 - create the evaluator and score the single, relevant tool call
tool_call_accuracy = ...
pprint(...)

# TODO 3.2 - add a second call about London and score both calls together
irrelevant_call = ...
pprint(...)

# TODO 3.3 - create TaskAdherenceEvaluator and score a vague answer vs a complete one
task_adherence_evaluator = ...

## Step 4 - Batch evaluation (~10 min)

Use the `batch_evaluation` helper from `lab_utils.py` on `assets/evaluation_data.jsonl` (5 records).
Keep `publish_to_foundry = False` for the first run, then try `True` if you have a Foundry project.

In [ ]:
from lab_utils import batch_evaluation

publish_to_foundry = False

# TODO 4.1 - batch run with tool_call_accuracy
local_path, run = batch_evaluation(
    eval_name=...,
    eval_object=...,
    eval_data_path="assets/evaluation_data.jsonl",
    eval_output_path="evaluation_results",
    publish_to_foundry=publish_to_foundry,
    foundry_project_endpoint=settings["foundry_project_endpoint"],
)

print(local_path)
pprint(run["metrics"])

# TODO 4.2 - repeat with task_adherence and compare the aggregated metrics

### Checkpoint - everything below is optional

## Step 5 (optional) - Groundedness and Response Completeness

Datasets: `assets/groundedness_data.jsonl` (`query`/`context`/`response`) and
`assets/response_completeness_data.jsonl` (`ground_truth`/`response`).

In [ ]:
from azure.ai.evaluation import GroundednessEvaluator, ResponseCompletenessEvaluator

# TODO 5.1 - score the "Alpine Explorer Tent" example: context says *second* most waterproof,
#            the response claims it is *the* most waterproof
# TODO 5.2 - batch evaluate both datasets

## Step 6 (optional) - Custom evaluators

* prompt-based: `assets/friendliness.prompty` + `assets/friend.py`
* code-based: `assets/response_length_score.py`

Optionally publish them to the Foundry evaluator catalog (needed by the optional part of Lab 03).

In [ ]:
from openai import AzureOpenAI
from azure.identity import get_bearer_token_provider
from friend import FriendlinessEvaluator
from response_length_score import ResponseLengthScoreEvaluator

# TODO 6.1 - build an AzureOpenAI client with get_bearer_token_provider(credential, "https://cognitiveservices.azure.com/.default")
# TODO 6.2 - score a warm answer and a rude one with FriendlinessEvaluator
# TODO 6.3 - score three answers of different length with ResponseLengthScoreEvaluator

## Step 7 (optional) - Content safety

`ViolenceEvaluator` / `SelfHarmEvaluator` are service-backed: they take `credential` and
`azure_ai_project=settings["foundry_project_endpoint"]`, no judge model.

Tip: in `azure-ai-evaluation 1.18.3` a refusal can return the metric name as `Violence` instead of
`violence`, which the SDK turns into an empty dict - the solution notebook shows the subclass that
normalizes it.

In [ ]:
from azure.ai.evaluation import ViolenceEvaluator, SelfHarmEvaluator

# TODO 7.1 - score a refusal and a compliant harmful answer, and compare